# AI Agent Security — Multi-Step Tool Attacks

Competition submission for **ai-agent-security-multi-step-tool-attacks**.

Writes `attack.py` (defining `AttackAlgorithm(AttackAlgorithmBase)`) to `/kaggle/working/`, then starts the hosted evaluation server. The evaluator replays the attack candidates against GPT-OSS and Gemma 4 under the public and private guardrails and produces `submission.csv`.

In [ ]:
import sys, glob
from pathlib import Path

sys.argv = [sys.argv[0]]  # avoid argparse conflicts in Kaggle

# The competition dataset ships kaggle_evaluation/ and aicomp_sdk/ at its root.
dataset_root = None
for candidate in glob.glob('/kaggle/input/**/kaggle_evaluation', recursive=True):
    dataset_root = str(Path(candidate).parent)
    if dataset_root not in sys.path:
        sys.path.insert(0, dataset_root)
    print('Dataset root:', dataset_root)
    break
assert dataset_root, 'competition dataset (kaggle_evaluation/) not found under /kaggle/input'
import aicomp_sdk  # noqa: F401  -- confirm SDK importable
print('Setup complete; aicomp_sdk importable')

In [ ]:
import base64, importlib.util

ATTACK_PY_B64 = '''\
...
'''

attack_src = base64.b64decode(ATTACK_PY_B64)
with open('/kaggle/working/attack.py', 'wb') as f:
    f.write(attack_src)
print('Wrote /kaggle/working/attack.py (%d bytes)' % len(attack_src))

spec = importlib.util.spec_from_file_location('user_attack_check', '/kaggle/working/attack.py')
_m = importlib.util.module_from_spec(spec)
spec.loader.exec_module(_m)
from aicomp_sdk.attacks import AttackAlgorithmBase
assert issubclass(_m.AttackAlgorithm, AttackAlgorithmBase)
print('attack.py validated: AttackAlgorithm subclasses AttackAlgorithmBase')

In [ ]:
# 'Save & Run All' only checks the notebook runs; real scoring happens during
# the competition rerun, which replays attack.py against the target models.
import kaggle_evaluation.jed_attack_134815.jed_attack_inference_server as server
server.JEDAttackInferenceServer().serve()